# Baseline retrieval test (AISIS)


In [ ]:
!pip install -q -U sentence-transformers

In [ ]:
import glob, time, gc, json, numpy as np, pandas as pd, torch
from sentence_transformers import SentenceTransformer

dev = pd.read_excel(glob.glob("/kaggle/input/**/AISIS_DevSet_v1.xlsx", recursive=True)[0], sheet_name="DevSet").fillna("")
kb = pd.read_excel(glob.glob("/kaggle/input/**/AISIS_KnowledgeBase_v1.xlsx", recursive=True)[0], sheet_name="KB")
passages, pids = kb.text.tolist(), kb.passage_id.tolist()
gold_idx = {p: i for i, p in enumerate(pids)}

ev = dev[dev.type.isin(["answerable", "high_risk"])]
rows = []
for r in ev.itertuples():
    rows.append((r.dev_id, r.split, "standard", "all", r.query_standard, gold_idx[r.passage_id]))
    rows.append((r.dev_id, r.split, "dialect", r.dialect_differs, r.query_dialect_synth, gold_idx[r.passage_id]))
Q = pd.DataFrame(rows, columns=["dev_id", "split", "version", "differs", "query", "gold"])
print(len(Q), "queries,", len(passages), "passages")   # expect 400 queries, 136 passages

In [ ]:
def evaluate(name, enc_q, enc_p, extra=None):
    P = enc_p(passages)
    torch.cuda.reset_peak_memory_stats()
    t0 = time.time(); QE = enc_q(Q["query"].tolist()); lat = 1000 * (time.time() - t0) / len(Q)
    S = QE @ P.T
    order = np.argsort(-S, axis=1)
    rank = np.array([int(np.where(order[i] == g)[0][0]) + 1 for i, g in enumerate(Q.gold)])
    out = Q.drop(columns=["query"]).assign(model=name, rank=rank)
    print(name, "| R@1 =", round((rank == 1).mean(), 3), "| R@3 =", round((rank <= 3).mean(), 3), "| ms/query", round(lat, 1))
    return out, dict(model=name, ms_per_query=lat, peak_gb=torch.cuda.max_memory_allocated() / 1e9, **(extra or {}))

CFG = {
 "bge-m3": ("BAAI/bge-m3", "", ""),
 "mE5-large": ("intfloat/multilingual-e5-large", "query: ", "passage: "),
 "mE5-small": ("intfloat/multilingual-e5-small", "query: ", "passage: "),
 "bangla-e5-small (prefix)": ("kazalbrur/bangla-embed-e5-small", "query: ", "passage: "),
 "bangla-e5-small (no prefix)": ("kazalbrur/bangla-embed-e5-small", "", ""),
}
ranks, meta = [], []
for key, (hf, qp, pp) in CFG.items():
    m = None
    try:
        m = SentenceTransformer(hf, device="cuda")
        def enc(pre, m=m):
            return lambda xs: m.encode([pre + x for x in xs], normalize_embeddings=True, batch_size=32, convert_to_numpy=True)
        o, mt = evaluate(key, enc(qp), enc(pp), dict(params_M=sum(p.numel() for p in m.parameters()) / 1e6))
        ranks.append(o); meta.append(mt)
    except Exception as e:
        print("FAILED", key, repr(e)[:400])
    finally:
        del m; gc.collect(); torch.cuda.empty_cache()

In [ ]:
# Our own model goes here later.
# def encode_own(texts): ...   # must return an L2-normalised numpy array of shape (n, dim)
# o, mt = evaluate("ours", encode_own, encode_own, dict(params_M=...)); ranks.append(o); meta.append(mt)

In [ ]:
R = pd.concat(ranks)
R.to_csv("/kaggle/working/ranks.csv", index=False)
pd.DataFrame(meta).to_csv("/kaggle/working/retrieval_meta.csv", index=False)

def summ(g):
    return pd.Series({"R@1": (g["rank"] == 1).mean(), "R@3": (g["rank"] <= 3).mean(), "MRR": (1 / g["rank"]).mean(), "n": len(g)})
sub = R[(R.version == "standard") | (R.differs == "yes")]
print(sub.groupby(["split", "version", "model"]).apply(summ).round(3).unstack("version").to_string())
print()
print(pd.DataFrame(meta).round(2).to_string())

In [ ]:
import base64
from IPython.display import HTML
links = []
for name in ["ranks.csv", "retrieval_meta.csv"]:
    data = base64.b64encode(open(f"/kaggle/working/{name}", "rb").read()).decode()
    links.append(f'<p><a download="{name}" href="data:text/csv;base64,{data}">Download {name}</a></p>')
HTML("".join(links))